In [1]:
import pandas as pd
df=pd.read_csv('/workspaces/BlizzardX/Data/cleaned_data.csv')

In [2]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [3]:
from src.Model.feature_engineering import FeatureEngineering
fe=FeatureEngineering(df)

In [4]:
df=fe.apply_all_features()

In [5]:
df.columns

Index(['DATE', 'Station_ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME',
       'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD', 'Station_Location',
       'Station_Lat_Long_Interaction', 'Day_of_Week', 'Day_of_Year',
       'Temp_Diff', 'Rolling_Mean_TMIN_7', 'Rolling_10thPercentile_TMIN_7',
       'TMIN_Rolling_30_Diff', 'EWMA_TMIN_7', 'Seasonal_TMIN_Anomaly',
       'Rolling_Max_TMIN_30', 'Rolling_Min_TMIN_30', 'EWMA_TMIN_30',
       'TMIN_Lag1', 'SnowyDay', 'SnowyDaysCount_7', 'Cumulative_SnowDepth_7',
       'Cumulative_Snowfall_Lag7', 'SNWD_Lag1', 'SNWD_Lag2',
       'Rolling_Sum_SNWD_7', 'SNWD_TMIN_Interaction', 'Snowfall_Intensity',
       'SNWD_Snowfall_Diff', 'PRCP_Lag1', 'PRCP_Lag2',
       'Cumulative_Precipitation_7', 'Rolling_Sum_PRCP_14',
       'TMAX_PRCP_Interaction', 'Rolling_Mean_TMIN_30',
       'TMIN_SNOW_Interaction'],
      dtype='object')

In [6]:
df.isnull().sum()

DATE                                 0
Station_ID                           0
LATITUDE                             0
LONGITUDE                            0
ELEVATION                            0
NAME                                 0
Season                           97381
TMIN                                 0
TMAX                                 0
PRCP                                 0
SNOW                                 0
SNWD                                 0
Station_Location                     0
Station_Lat_Long_Interaction         0
Day_of_Week                          0
Day_of_Year                          0
Temp_Diff                            0
Rolling_Mean_TMIN_7                  6
Rolling_10thPercentile_TMIN_7        6
TMIN_Rolling_30_Diff                 6
EWMA_TMIN_7                          0
Seasonal_TMIN_Anomaly                0
Rolling_Max_TMIN_30                 29
Rolling_Min_TMIN_30                 29
EWMA_TMIN_30                         0
TMIN_Lag1                

In [6]:
# List of columns to exclude from rounding
exclude_columns = ['DATE', 'ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season']

# Get all columns in the DataFrame
all_columns = df.columns

# Select numerical columns excluding the ones listed above
columns_to_round = [col for col in all_columns if col not in exclude_columns]

# Round the selected columns to 2 decimal places
df[columns_to_round] = df[columns_to_round].round(2)


In [25]:
df.to_csv('/workspaces/BlizzardX/Data/Final_Processed.csv', index=False)  # Save the processed DataFrame

In [29]:
df.columns  # Display the columns in the DataFrame

Index(['DATE', 'ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season',
       'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD', 'Week',
       'TMIN_10thPercentile_Week', 'Temp_Diff', 'SnowyDay', 'SNWD_Lag1',
       'SNWD_Lag2', 'SnowyDaysCount_7', 'Cumulative_SnowDepth_7', 'PRCP_Lag1',
       'PRCP_Lag2', 'Cumulative_Precipitation_7', 'Rolling_Mean_TMIN_7',
       'Rolling_10thPercentile_TMIN_7', 'Day_of_Week', 'Day_of_Year',
       'SNWD_TMIN_Interaction', 'Snowfall_Intensity',
       'Seasonal_TMIN_Deviation', 'Rolling_Mean_TMIN_30',
       'TMIN_Rolling_30_Diff', 'EWMA_TMIN_7', 'Previous_Season_TMIN',
       'Rolling_Sum_PRCP_14', 'Cumulative_Snowfall_Lag7',
       'TMIN_SNOW_Interaction', 'TMAX_PRCP_Interaction',
       'Seasonal_TMIN_Anomaly', 'Days_Since_Last_Precip',
       'Rolling_Max_TMIN_30', 'Rolling_Min_TMIN_30', 'EWMA_TMIN_30',
       'SNWD_Snowfall_Diff', 'TMIN_Lag1', 'TMAX_Lag1', 'SNOW_Lag1',
       'Rolling_Sum_SNWD_7'],
      dtype='object')

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into train and test (e.g., 80% train, 20% test)
train_size = int(len(df) * 0.8)
train, test = df[:train_size], df[train_size:]

# Scaling the features (for traditional models)
scaler = StandardScaler()
X_train = train.drop(columns=['DATE', 'ID','NAME', 'Season', 'TMIN'])  # Assuming TMIN is the target variable
X_test = test.drop(columns=['DATE', 'ID','NAME',  'Season', 'TMIN'])

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Target variable (TMIN)
y_train = train['TMIN']
y_test = test['TMIN']


In [34]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Ensure DATE column is in datetime format
df['DATE'] = pd.to_datetime(df['DATE'])

# Set 'DATE' as the index
df.set_index('DATE', inplace=True)

# Optionally, set frequency (e.g., daily)
df = df.asfreq('D')  # Change 'D' to the appropriate frequency ('W' for weekly, etc.)

# Select your target variable (e.g., 'TMIN')
y_train = df['TMIN'][:len(df)//2]  # First half for training
y_test = df['TMIN'][len(df)//2:]  # Second half for testing

# ARIMA model (adjust the order as needed)
arima_model = ARIMA(y_train, order=(5, 1, 0))
arima_model_fit = arima_model.fit()

# Forecasting on test data
arima_forecast = arima_model_fit.forecast(steps=len(y_test))

# Now arima_forecast is a pandas Series with forecasts


In [35]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Define and fit SARIMA model
sarima_model = SARIMAX(y_train, order=(5, 1, 0), seasonal_order=(1, 1, 0, 7))  # You can adjust these orders
sarima_model_fit = sarima_model.fit()

# Forecasting on test data
sarima_forecast = sarima_model_fit.forecast(steps=len(y_test))

# Print and visualize forecast
print(sarima_forecast)


1986-08-02      9.520762
1986-08-03     10.790201
1986-08-04      8.820720
1986-08-05      7.484025
1986-08-06      6.835082
                 ...    
2025-02-27    287.427188
2025-02-28    288.272964
2025-03-01    288.909985
2025-03-02    289.746125
2025-03-03    288.442222
Freq: D, Name: predicted_mean, Length: 14094, dtype: float64
